# Mental Health Assistant

Interactive notebook for testing RAG chatbot inference

In [1]:
from src.rag_pipeline import RagPipeline
from src.generator import QwenGenerator
import json

/home/gusevsaint/Workspace/study/practice/mental-helper/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Models and Data

In [3]:
rag = RagPipeline()
generator = QwenGenerator()

with open("data/qa_dataset.json", "r", encoding="utf-8") as f:
    qa_data = json.load(f)

print(f"Dataset loaded: {len(qa_data)} Q&A pairs")

llama_context: n_ctx_per_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Dataset loaded: 172 Q&A pairs


## Generate Answer

In [ ]:
def answer(query, rag_enabled=True):

    if rag_enabled:
        retrieved = rag.retrieve(query, top_k=3)
        context = "\n".join(
            f"[{i}] Q: {r['question']}\nA: {r['answer']}"
            for i, r in enumerate(retrieved, 1)
        )[:2500]
        generated = generator.generate(query, context=context, max_tokens=256)
    else:
        retrieved = []
        generated = generator.generate(query, context="", max_tokens=256)

    return {
        "answer": generated,
        "sources": (
            [{"question": r["question"], "score": r["score"]} for r in retrieved]
            if rag_enabled
            else []
        ),
        "query": query,
        "rag_enabled": rag_enabled,
    }

In [8]:
question = "What is a panic attack?"

result = answer(question, rag_enabled=True)

print("Query:", result["query"])
print("\nAnswer:")
print(result["answer"])

if result["sources"]:
    print("\nSources:")
    for i, src in enumerate(result["sources"], 1):
        print(f"{i}. {src['question'][:60]}... (relevance: {src['score']:.3f})")

Query: What is a panic attack?

Answer:
A panic attack is a sudden and intense surge of fear or discomfort that reaches its peak within minutes. These attacks can involve heart palpitations, shortness of breath, sweating, or feeling of choking. They are often accompanied by physical symptoms like a racing heartbeat, shortness of breath, or nausea. Panic attacks can happen to anyone, but having more than one may be a sign of panic disorder, a mental health condition characterized by sudden and repeated panic attacks.

Sources:
1. What is a panic attack?... (relevance: 0.942)
2. What are symptoms of panic attack vs. anxiety attack?... (relevance: 0.889)
3. I have been experiencing a sudden increase in panic attacks.... (relevance: 0.880)
